In [2]:
import gffutils
import pyfaidx
from Bio.Seq import Seq
import subprocess
from Bio import SeqIO

from Bio import Phylo

import pandas as pd
import json


In [3]:
def remove_zero_length_branches(tree):
    """
    Removes clades with a branch_length of 0 from a Biopython Tree object.
    This effectively prunes zero-length branches and creates polytomies.
    """
    # Create a list to store clades to be processed (children of zero-length clades)
    clades_to_process = []

    # Iterate through all clades in the tree
    for clade in tree.find_clades():
        # Check if the clade has children and a branch_length of 0
        if clade.branch_length == 0 and clade.clades:
            clades_to_process.append(clade)
    print(f"Removing {len(clades_to_process)} zero-length branches")
    # Process clades with zero-length branches
    for zero_clade in clades_to_process:
        parent = tree.get_path(zero_clade)[-2]  # Get the parent of the zero-length clade
        
        if parent:
            # Remove the zero-length clade from its parent's children
            parent.clades.remove(zero_clade)
            # Add the children of the zero-length clade directly to the parent
            parent.clades.extend(zero_clade.clades)

    return tree

def prune_tree_to_seqs(wgstree, fastaf, pruned_tree_file):
    """
    Prune a tree to only include tips present in the fasta file.
    Args:
        wgstree (str): Path to the input tree file (Newick format).
        fastaf (str): Path to the fasta file containing sequence names to keep.
        pruned_tree_file (str): Path to write the pruned tree.
    """
    seq_names = set(SeqIO.to_dict(SeqIO.parse(fastaf, "fasta")).keys())
    
    # Read the tree
    tree = Phylo.read(wgstree, "newick")
    tips = [c.name for c in tree.get_terminals()]
    # Find tips to prune (not in fasta)
    tips_to_prune = [term for term in tips if term not in seq_names]
    
    # Prune tips
    if(len(tips_to_prune) == len(tips)):
        print(len(tips), len(seq_names), 0)
        exit(1)
    else:
        for tip in tips_to_prune:
            tree.prune(tip)    
        #remove zero-length branches
        nozerotree = remove_zero_length_branches(tree) 

        # Write pruned tree
        print(len(tips), len(seq_names), len(tree.get_terminals()))

        Phylo.write(nozerotree, pruned_tree_file, "newick")
    return(len(tree.get_terminals()))

def filter_seqs_to_tree(fastaf, treefile, output_fasta):
    """
    Filter sequences in a FASTA file to only those present as tips in a tree file.
    Args:
        fastaf (str): Path to input FASTA file.
        treefile (str): Path to tree file (Newick format).
        output_fasta (str): Path to output filtered FASTA file.
    """
    # Get tip names from tree
    tree = Phylo.read(treefile, "newick")
    tip_names = set([term.name for term in tree.get_terminals()])
    # Parse FASTA and filter
    records = [rec for rec in SeqIO.parse(fastaf, "fasta")]
    nrecords = len(records)
    goodrecords = [rec for rec in records if rec.id in tip_names]
    # Write filtered FASTA
    with open(output_fasta, "w") as out:
        SeqIO.write(goodrecords, out, "fasta")

    print(nrecords, len(tip_names), len(goodrecords))
    return(len(goodrecords))

In [4]:
def prune_tree_to_samples(wgstree, samples, pruned_tree_file):
    """
    Prune a tree to only include tips present in the samples file.
    Args:
        wgstree (str): Path to the input tree file (Newick format).
        samples (str): Path to the samples file (one sample name per line).
        pruned_tree_file (str): Path to write the pruned tree.
    """
    with open(samples) as f:
        sample_names = set(line.strip() for line in f if line.strip())
    
    # Read the tree
    tree = Phylo.read(wgstree, "newick")
    tips = [c.name for c in tree.get_terminals()]
    # Find tips to prune (not in samples)
    tips_to_prune = [term for term in tips if term not in sample_names]
    
    # Prune tips
    if(len(tips_to_prune) == len(tips)):
        print(len(tips), len(sample_names), 0)
        exit(1)
    else:
        for tip in tips_to_prune:
            tree.prune(tip)    
        #remove zero-length branches
        nozerotree = remove_zero_length_branches(tree) 

        # Write pruned tree
        print(len(tips), len(sample_names), len(tree.get_terminals()))
        Phylo.write(nozerotree, pruned_tree_file, "newick")
    return(len(tree.get_terminals()))

def filter_seqs_to_samples(fastaf, samples, output_fasta):
    """
    Filter sequences in a FASTA file to only those present in the samples file.
    Args:
        fastaf (str): Path to input FASTA file.
        samples (str): Path to the samples file (one sample name per line).
        output_fasta (str): Path to output filtered FASTA file.
    """
    with open(samples) as f:
        sample_names = set(line.strip() for line in f if line.strip())
    
    # Parse FASTA and filter
    records = [rec for rec in SeqIO.parse(fastaf, "fasta")]
    nrecords = len(records)
    goodrecords = [rec for rec in records if rec.id in sample_names]
    # Write filtered FASTA
    with open(output_fasta, "w") as out:
        SeqIO.write(goodrecords, out, "fasta")

    print(nrecords, len(sample_names), len(goodrecords))
    return(len(goodrecords))

In [5]:
def remove_duplicate_sequences(fasta_file, output_file):
    """
    Reads a FASTA file, removes duplicate sequences (keeping the one with the lowest alphanumeric ID),
    and writes the unique sequences to output_file.
    """

    seq_dict = {}
    for record in SeqIO.parse(fasta_file, "fasta"):
        seq_str = str(record.seq)
        if seq_str not in seq_dict or record.id < seq_dict[seq_str].id:
            seq_dict[seq_str] = record

    with open(output_file, "w") as out:
        SeqIO.write(seq_dict.values(), out, "fasta")

In [41]:
def all_descendants(clade):
    descendants = []
    for child_clade in clade.clades:
        descendants.append(child_clade)
        descendants.extend(all_descendants(child_clade))
    return descendants


#annotate tree with clade assignments
def annotate_tree(treefile,outtree,clades,clade=None,strategy="MRCA"):
    tree = Phylo.read(treefile, "newick")
    if clade is not None: cladenames = [clade]
    else: cladenames = clades.keys()
    
    for C in cladenames:
        cladelist = clades[C]
        print(C, len(clades[C]))
        tip_names = set([term.name for term in tree.get_terminals()])
        goodtips = [rec for rec in cladelist if rec in tip_names]
        print(f"Annotating clade {C} with {len(goodtips)} tips in length {len(tip_names)} tree")
        
        tag = '{Foreground}'
        mrca = tree.common_ancestor(goodtips)
        # Set the color and width of the branch leading to this clade
        cladestr = '{'+f'{C}'+'}'
        if  strategy == "MRCA":  # Tag MRCA
            mrca.name = f'{mrca.name}{cladestr}{tag}'
        if  strategy == "tips":  # Tag tips
            for C in [c for c in tree.get_terminals() if c.name in goodtips]:
                cladestr = '{'+f'{C}'+'}'
                C.name = f'{C.name}{cladestr}{tag}'
        if strategy == "path":  # Tag path from tips to MRCA
            pathnodes = {}
            for tip in goodtips:
                #append only new nodes in path to list of paths
                for n in get_path_to_ancestor(tree, tip, mrca):
                    if n.name not in pathnodes:
                        pathnodes[n.name] = n
            pathnodes = [pathnodes[p] for p in pathnodes]
            for p in pathnodes:
                cladestr = '{'+f'{C}'+'}'
                for n in p:
                    n.name = f'{n.name}{cladestr}{tag}'
            print(f"Marking {len(goodtips)} of {len(tree.get_terminals())} ({len(allpaths)} nodes, mrca {mrca.name})")
        if  strategy == "all":  # Tag MRCA and all descendants
            mrca.name = f'{mrca.name}{cladestr}{tag}'
            descs = all_descendants(mrca)
            for dclade in descs:
                cladestr = '{'+f'{C}'+'}'
                dclade.name = f'{dclade.name}{cladestr}{tag}'
        # Draw the tree (e.g., to ASCII or using Matplotlib)
        #Phylo.draw_ascii(tree)
            print(f"Marking {len(goodtips)} of {len(tree.get_terminals())} ({len(descs)} nodes, mrca {mrca.name})")
    Phylo.write(tree, outtree, "newick")
    return len(goodtips)


# Define a function to find the path from a tip to an ancestor
def get_path_to_ancestor(tree, tip, ancestor):
    path_to_root = tree.get_path(tip)
    path_to_mrca = []
    # Iterate through the path from root and stop at the ancestor
    for clade in reversed(path_to_root):
        path_to_mrca.append(clade)
        if clade == ancestor:
            break
    return path_to_mrca




In [42]:
# Run RELAX using HyPhy on the codon alignment and IQ-TREE treefile


def run_relax(codontrimf, treefile, relaxout):
    """Run RELAX using HyPhy on the codon alignment and IQ-TREE treefile."""
    errfile = relaxout + ".err"    
    outfile = relaxout + ".out"    
    command = [
        "hyphy", "relax",
        "--alignment", codontrimf,
        "--tree", treefile,
        "--output", relaxout,
        "--models", "Minimal",
        "--test", "Foreground"
    ]
    with open(errfile, "w") as errfile, open(outfile, "w") as outfile:
        print(" ".join(command))
        subprocess.run(command, check=True,stderr=errfile, stdout=outfile)

def parse_relax_json(relax_json_file):
    """
    Parse RELAX JSON output and extract all values within 'test results'.
    Returns a dictionary of the extracted values.
    """
    with open(relax_json_file) as f:
        data = json.load(f)
    test_results = data.get('test results')
    print(test_results)
    LR = test_results.get('LRT')
    p = test_results.get('p-value')
    K = test_results.get('relaxation or intensification parameter')
    return LR, p, K


targets=["RSVA", "RSVB"]
genes = ["G", "F", "L", "N", "P", "M2-1", "M2-2", "M", "SH", "NS1", "NS2"]

targets=["RSVB"]
#genes = ["G"]

results = pd.DataFrame(columns=["target","gene","clade","n_tips","LR","p","K"])

for target in targets:

    nextcladeF = f'{target}_nextclade.tsv'
    df_clades = pd.read_csv(nextcladeF, usecols=['seqName', 'clade', 'qc.overallStatus'], sep="\t")
    # Add 'majorclade' column by removing everything after the second '.' in 'clade'
    df_clades['majorclade'] = df_clades['clade'].str.split('.').str[:4].str.join('.')
    #retain only sequences with qc = 'good'
    #df_clades = df_clades[df_clades['qc.overallStatus'].str.contains('good', na=False)]
    #clades = df_clades.groupby('majorclade')['seqName'].apply(list).to_dict()
    clades = df_clades.groupby('clade')['seqName'].apply(list).to_dict()
    for c in clades:
        print(c, len(clades[c]))
    print("")

    for gene in genes:
        codonf = f"geneseqs/{target}_{gene}_codonaln_PGCOE.fasta"             #nucleotide codon alignment
        codonfuniq = f"geneseqs/{target}_{gene}_codonaln_PGCOE_uq.fasta"             #nucleotide codon alignment

        wgstree = f"geneseqs/{target}_{gene}_tree_pruned_PGCOE.nwk"
        wgstreeuniq = f"geneseqs/{target}_{gene}_tree_pruned_PGCOE_uq.nwk"

        prune_tree_to_seqs(wgstree, codonf, wgstreeuniq)
        remove_duplicate_sequences(codonf, codonfuniq)
        prune_tree_to_seqs(wgstreeuniq, codonfuniq, wgstreeuniq)
        filter_seqs_to_tree(codonfuniq, wgstreeuniq, codonfuniq)
        
        for clade in clades.keys():
            wgstrim = wgstree.replace(".nwk", f"_{clade}.nwk")
            ntips = annotate_tree(wgstreeuniq,wgstrim,clades,clade=clade,strategy="path")
            if(ntips >= 5):
                #hyphy outputs
                relaxout = f"hyphy/{target}_{gene}_{clade}_relax.json"                  #RELAX output
                #absrelcsv = f"hyphy/{target}_{gene}_absrel_table.csv"                  #RELAX table output
                run_relax(codonfuniq, wgstrim, relaxout)
                LR, p, K = parse_relax_json(relaxout)

            else: 
                print(f"Skipping {target} {gene} {clade} with {ntips} tips")
                LR, p, K = None, None, None
            results = pd.concat([results, pd.DataFrame([[target, gene, clade, ntips, LR, p, K]], 
                                                       columns=["target","gene","clade","n_tips","LR","p","K"])], ignore_index=True)
    results.to_csv(f'hyphy/{target}_RELAX_results_by_clade.csv', index=False)




B.D.4.1 5
B.D.4.1.1 45
B.D.4.1.3 49
B.D.E.1 389
B.D.E.1.1 7
B.D.E.1.2 14
B.D.E.1.3 7
B.D.E.1.4 4
B.D.E.1.7 3
B.D.E.1.8 10
B.D.E.5 2
B.D.E.6 1
B.D.E.7 1

Removing 0 zero-length branches
344 344 344
Removing 0 zero-length branches
344 234 234
234 234 234
B.D.4.1 5
Annotating clade B.D.4.1 with 5 tips in length 234 tree


NameError: name 'allpaths' is not defined

In [24]:
annotate_tree(wgstreeuniq,"mytree.B.D.4.1.nwk",clades,clade="B.D.4.1",MRCA_only=False,tips_only=True)

B.D.4.1 56
Annotating clade B.D.4.1 with 24 tips in length 234 tree


24

In [ ]:
# Run ABSREL using HyPhy on the codon alignment and IQ-TREE treefile


def run_absrel(codontrimf, treefile, absrelout):
    """Run aBSREL using HyPhy on the codon alignment and IQ-TREE treefile."""
    command = [
        "hyphy", "absrel",
        "--alignment", codontrimf,
        "--tree", treefile,
        "--output", absrelout,
        "--branches", "Foreground"
    ]
    print(" ".join(command))
    subprocess.run(command, check=True)

    

target="RSVA"
gene="G"
wgstree = f"geneseqs/{target}_{gene}_tree_pruned.nwk"
wgstrim = wgstree.replace(".nwk", f"_annotated.nwk")

ntips = annotate_tree(wgstree,wgstrim,clades)

codonf = f"geneseqs/{target}_{gene}_codonaln.fasta"             #nucleotide codon alignment

if(ntips >= 5):
    #hyphy outputs
    absrelout = f"hyphy/{target}_{gene}_absrel.json"                  #FUBAR output
    #absrelcsv = f"hyphy/{target}_{gene}_absrel_table.csv"                  #FUBAR table output
            
    #run_absrel(codonf, wgstrim, absrelout)


